- Author: Yousef Al Zeer

# Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingRegressor # NEW
from sklearn.ensemble import RandomForestRegressor # NEW
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import root_mean_squared_error

def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = root_mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics



def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Loading Data

In [ ]:
fpath = "/content/drive/MyDrive/Intro to ML /Week 2 /Core/Boston_Housing_from_Sklearn - Boston_Housing_from_Sklearn.csv"
df = pd.read_csv(fpath)
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   NOX      506 non-null    float64
 2   RM       506 non-null    float64
 3   AGE      506 non-null    float64
 4   PTRATIO  506 non-null    float64
 5   LSTAT    506 non-null    float64
 6   PRICE    506 non-null    float64
dtypes: float64(7)
memory usage: 27.8 KB


,CRIM,NOX,RM,AGE,PTRATIO,LSTAT,PRICE
0,0.00632,0.538,6.575,65.2,15.3,4.98,24.0
1,0.02731,0.469,6.421,78.9,17.8,9.14,21.6
2,0.02729,0.469,7.185,61.1,17.8,4.03,34.7
3,0.03237,0.458,6.998,45.8,18.7,2.94,33.4
4,0.06905,0.458,7.147,54.2,18.7,5.33,36.2


# Train-Test Split

In [ ]:
target = 'PRICE'

y = df[target].copy()
X = df.drop(columns=target).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

# Deafult Model

In [ ]:
bagreg = BaggingRegressor(random_state = 42)
bagreg.fit(X_train, y_train)
# Call custom function for evaluation
evaluate_regression(bagreg, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.103
- MSE = 3.487
- RMSE = 1.867
- R^2 = 0.961

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.316
- MSE = 12.575
- RMSE = 3.546
- R^2 = 0.820


#Tune the Bagged Trees

In [ ]:
bagreg.get_params()

{'bootstrap': True,
 'bootstrap_features': False,
 'estimator': None,
 'max_features': 1.0,
 'max_samples': 1.0,
 'n_estimators': 10,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
}

gridcv = GridSearchCV(bagreg , param_grid , n_jobs=-1 , verbose=1)
gridcv.fit(X_train , y_train)


Fitting 5 folds for each of 18 candidates, totalling 90 fits


GridSearchCV(estimator=BaggingRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_features': [0.5, 0.7, 1.0],
                         'max_samples': [0.5, 0.7, 1.0],
                         'n_estimators': [100, 200]},
             verbose=1)

In [ ]:
gridcv.best_params_

{'max_features': 1.0, 'max_samples': 0.7, 'n_estimators': 100}

In [ ]:
best_bagreg_grid = gridcv.best_estimator_
evaluate_regression(best_bagreg_grid , X_train , y_train , X_test , y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.282
- MSE = 3.808
- RMSE = 1.951
- R^2 = 0.957

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.143
- MSE = 12.276
- RMSE = 3.504
- R^2 = 0.825


# RandomForest Default

In [ ]:
rf = RandomForestRegressor(random_state = 42)
rf.fit(X_train , y_train)

RandomForestRegressor(random_state=42)

In [ ]:
evaluate_regression(rf,  X_train , y_train , X_test , y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 0.954
- MSE = 2.028
- RMSE = 1.424
- R^2 = 0.977

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.208
- MSE = 11.635
- RMSE = 3.411
- R^2 = 0.834


# Tune Random Forest

In [ ]:
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [ ]:
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 8, 10, 12],
    'min_samples_leaf': [5, 10, 20],
    'max_features': [0.5, 0.7, 'sqrt'],
    'bootstrap': [True]
}

gridcv = GridSearchCV(rf , rf_params , n_jobs=-1 , verbose=1)
gridcv.fit(X_train , y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


GridSearchCV(estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'bootstrap': [True], 'max_depth': [5, 8, 10, 12],
                         'max_features': [0.5, 0.7, 'sqrt'],
                         'min_samples_leaf': [5, 10, 20],
                         'n_estimators': [100, 200]},
             verbose=1)

In [ ]:
gridcv.best_params_

{'bootstrap': True,
 'max_depth': 12,
 'max_features': 0.5,
 'min_samples_leaf': 5,
 'n_estimators': 100}

In [ ]:
best_rf_model = gridcv.best_estimator_
evaluate_regression(best_rf_model,  X_train , y_train , X_test , y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.825
- MSE = 8.331
- RMSE = 2.886
- R^2 = 0.906

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.189
- MSE = 12.946
- RMSE = 3.598
- R^2 = 0.815


# Comprsion: Random Forest VS Bagged Trees

### Comparison Results

| Model | Configuration | Train $R^2$ | Test $R^2$ | MAE | RMSE |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Bagged Trees** | Tuned | 0.957 | 0.825 | 2.143 | 3.504 |
| **Random Forest** | Tuned | 0.906 | 0.815 | 2.189 | 3.598 |



Although Bagged Trees model achieved a slightly higher $R^2$ on test set, Random Forest model is the superior choice for deployment. This is because it provides a much better balance between training and testing performance, with a significantly smaller gap (9% vs 13%)


***Random Forest*** is the choice here

Parameters that achieved these results are:

* **n_estimators:** 100
* **max_depth:** 12
* **max_features:** 0.5
* **min_samples_leaf:** 5
* **bootstrap:** True


- Stakeholders can expect the predictions to be, on average within **$2,189** of the actual house price.

- With a relative error of approximately 10.7%, the model achieves an accuracy of nearly 90% relative to market values. This helps stakeholders develop a reliable risk management plan.
